In [ ]:
from netCDF4 import Dataset, MFDataset
import numpy as np
import numpy.ma as ma
import matplotlib.pyplot as plt
import xarray as xr 
from mpl_toolkits.basemap import Basemap
import seaborn as sns
import dask
import string
from matplotlib import rc
from scipy import stats
from tqdm import tqdm
import math
import pandas as pd
import warnings
import matplotlib.patches as patches
from matplotlib.patches import Rectangle
import matplotlib.patches as mpatches
from sklearn import linear_model
import statsmodels.api as sm
import matplotlib.lines as mlines
import matplotlib
from matplotlib.pyplot import gca

warnings.filterwarnings('ignore')

In [ ]:
def import_file(path, mf):
    
    #set mf=True if more than one ensemble member is available
    
    dask.config.set(**{'array.slicing.split_large_chunks': False})
    
    if mf == True: 
        nc_fid=xr.open_mfdataset(path, concat_dim='realization', combine='nested')
    if mf == False: 
        nc_fid=xr.open_dataset(path)
    if mf == 'semi': 
        nc_fid=xr.open_mfdataset(path)
        
    return nc_fid

In [ ]:
def calc_polcap(var):
    
    if var.lat[0]<var.lat[1]: var=var.sel(lat=slice(60,90))
    else: var=var.sel(lat=slice(90,60))

    weights = np.cos(np.deg2rad(var.lat))
    var = var.weighted(weights)     
    var=var.mean(dim='lat')
    
    if 'lon' in var.dims: var=var.mean(dim='lon')
    
    return var

In [ ]:
def sel_spring(var):
    
        var=var.sel(time=var.time.dt.month.isin([3,4]))
        var=var.groupby("time.year").mean("time")
        
        return var

In [ ]:
label=[]
colors=[]
nc_fid_O3=[]


#WACCM (this study)

nc_fid_O3.append(import_file('/net/hydro/chemie/mfriedel/Data/WACCM/RCP8.5/all_members/O3/BRCP85WCN.e122.f19_g16.00*.cam.h0.O3.ml.nc', mf=True))

label.append('WACCM (this study)')
colors.append('red')

In [ ]:
#WACCM forcing file

nc_fid_O3.append(import_file('/net/hydro/chemie/mfriedel/Data/WACCM/ghg_forcing_RCP85_2005-2099_CMIP5_EnsMean.c140411.nc', mf=False))

label.append('WACCM forcing file')
colors.append('blue')

In [ ]:
nc_fid_O3.append(import_file('/net/hydro/chemie/mfriedel/Data/ccmi/RCPs/senC2rcp85/zmo3/UMSLIMCAT/r1i1p1/v1/zmo3_monthly_UMSLIMCAT_senC2rcp85_r1i1p1_200001-209912.nc', mf=False))
nc_fid_ps=import_file('/net/hydro/chemie/mfriedel/Data/ccmi/RCPs/senC2rcp85/ps_monthly_UMSLIMCAT_senC2rcp85_r1i1p1_200001-209912.nc', mf=False)

label.append('UMSLIMCAT')
colors.append('green')

In [ ]:
model_levels_waccm=nc_fid_O3[1]['lev'] #waccm model levels

o3_umslimcat=nc_fid_O3[2]['zmo3']

o3_umslimcat=o3_umslimcat.interp(plev=model_levels_waccm*100) #interpolate UMSLIMCAT pressure levels to WACCM model levels

o3_umslimcat=o3_umslimcat.interp(lat=nc_fid_O3[1].lat) #interpolate UMSLIMCAT to the same horizontal grid

o3_umslimcat_plot=sel_spring((calc_polcap(o3_umslimcat)))

In [ ]:
nc_fid_O3.append(import_file('/net/hydro/chemie/mfriedel/Data/ccmi/SOCOL_RCP85/zmo3_monthly_SOCOL3_senC2rcp85_r1i1p1_200001-209912.nc', mf=False))

label.append('SOCOL CCMI')
colors.append('orange')

In [ ]:
model_levels_waccm=nc_fid_O3[1]['lev'] #waccm model levels

o3_socol=nc_fid_O3[3]['zmo3']

o3_socol=o3_socol.interp(plev=model_levels_waccm*100) #interpolate SOCOL pressure levels to WACCM model levels

o3_socol=o3_socol.interp(lat=nc_fid_O3[1].lat) #interpolate SOCOL to the same horizontal grid

o3_socol_plot=sel_spring((calc_polcap(o3_socol)))

In [ ]:
O3_col=[]

for i in range(len(nc_fid_O3)-2):
    
    file=nc_fid_O3[i]
    if 'O3' in file.variables: var=file['O3']
    if 'zmo3' in file.variables: var=file['zmo3']

    O3_col.append(sel_spring((calc_polcap(var))))

O3_col.append(sel_spring((calc_polcap(o3_umslimcat))))
O3_col.append(sel_spring((calc_polcap(o3_socol))))



In [ ]:
fig, ax = plt.subplots(1,1,figsize=(5,8), constrained_layout=True, edgecolor='k',sharex='col')

for i in tqdm(range(len(O3_col))):
        
        if 'realization' in O3_col[i].dims: O3_x=O3_col[i].mean(dim='realization') 
        else: O3_x=O3_col[i]
        
        if 'lev' in O3_x.dims: O3_x=O3_x.sel(lev=100, method='nearest')
        if 'plev' in O3_x.dims: O3_x=O3_x.sel(plev=10000, method='nearest')
        
        O3_x.plot(color=colors[i], label=label[i], linestyle='-')
 
        if i==2:    
            slope, intercept, r_trends, p, se = stats.linregress(O3_x.year, O3_x)
            x=np.linspace(O3_x.year.min(), O3_x.year.max(), 100)
            plt.plot(x, x*slope +intercept, color='green', linestyle='-')
        if i==3:    
            slope, intercept, r_trends, p, se = stats.linregress(O3_x.year, O3_x)
            x=np.linspace(O3_x.year.min(), O3_x.year.max(), 100)
            plt.plot(x, x*slope +intercept, color='orange', linestyle='-')
           
    
           

for i in tqdm(range(len(O3_col))):
        
        if 'realization' in O3_col[i].dims: O3_x=O3_col[i].mean(dim='realization') 
        else: O3_x=O3_col[i]
        
        if 'lev' in O3_x.dims: O3_x=O3_x.sel(lev=70, method='nearest')
        if 'plev' in O3_x.dims: O3_x=O3_x.sel(plev=7000, method='nearest')
        
        O3_x.plot(color=colors[i], linestyle='-.')
        if i==2:    
            slope, intercept, r_trends, p, se = stats.linregress(O3_x.year, O3_x)
            x=np.linspace(O3_x.year.min(), O3_x.year.max(), 100)
            plt.plot(x, x*slope +intercept, color='green', linestyle='-')
        if i==3:    
            slope, intercept, r_trends, p, se = stats.linregress(O3_x.year, O3_x)
            x=np.linspace(O3_x.year.min(), O3_x.year.max(), 100)
            plt.plot(x, x*slope +intercept, color='orange', linestyle='-')
           
    
           
        
for i in tqdm(range(len(O3_col))):
        
        if 'realization' in O3_col[i].dims: O3_x=O3_col[i].mean(dim='realization') 
        else: O3_x=O3_col[i]
        
        if 'lev' in O3_x.dims: O3_x=O3_x.sel(lev=30, method='nearest')
        if 'plev' in O3_x.dims: O3_x=O3_x.sel(plev=3000, method='nearest')
        
        O3_x.plot(color=colors[i], linestyle=':')
        if i==2:    
            slope, intercept, r_trends, p, se = stats.linregress(O3_x.year, O3_x)
            x=np.linspace(O3_x.year.min(), O3_x.year.max(), 100)
            plt.plot(x, x*slope +intercept, color='green', linestyle='-')
        if i==3:    
            slope, intercept, r_trends, p, se = stats.linregress(O3_x.year, O3_x)
            x=np.linspace(O3_x.year.min(), O3_x.year.max(), 100)
            plt.plot(x, x*slope +intercept, color='orange', linestyle='-')
           
    
           
        
for i in tqdm(range(len(O3_col))):
        
        if 'realization' in O3_col[i].dims: O3_x=O3_col[i].mean(dim='realization') 
        else: O3_x=O3_col[i]
        
        if 'lev' in O3_x.dims: O3_x=O3_x.sel(lev=10, method='nearest')
        if 'plev' in O3_x.dims: O3_x=O3_x.sel(plev=1000, method='nearest')
        
        O3_x.plot(color=colors[i], linestyle='--')
        if i==2:    
            slope, intercept, r_trends, p, se = stats.linregress(O3_x.year, O3_x)
            x=np.linspace(O3_x.year.min(), O3_x.year.max(), 100)
            plt.plot(x, x*slope +intercept, color='green', linestyle='-')
        if i==3:    
            slope, intercept, r_trends, p, se = stats.linregress(O3_x.year, O3_x)
            x=np.linspace(O3_x.year.min(), O3_x.year.max(), 100)
            plt.plot(x, x*slope +intercept, color='orange', linestyle='-')
           
    
        
plt.legend()

In [ ]:
# adapt WACCM to UMSLIMCAT ozone trends

model_levels_waccm=nc_fid_O3[1]['lev'] #waccm model levels

#linearly detrend waccm ozone forcing file for each time of the year

o3_umslimcat=nc_fid_O3[2]['zmo3']
o3_umslimcat=o3_umslimcat.squeeze()
o3_umslimcat=o3_umslimcat.interp(plev=model_levels_waccm*100) #interpolate UMSLIMCAT pressure levels to WACCM model levels

o3_umslimcat=o3_umslimcat.interp(lat=nc_fid_O3[1].lat) #interpolate UMSLIMCAT to the same horizontal grid


# substract mean values
o3_umslimcat=o3_umslimcat.groupby('time.month')-o3_umslimcat.groupby('time.month').mean()


#o3_umslimcat=o3_umslimcat.where(o3_umslimcat.plev>300, other=None, drop=False) # mask values in troposphere

    
def calc_trend(group): 
    p = group.polyfit(dim='time', deg=1)
    fit=xr.polyval(group.time, p.polyfit_coefficients)
    return fit
    
o3_umslimcat=o3_umslimcat.groupby('time.month').apply(calc_trend)


    
def detrend(group):
    p = group.polyfit(dim='time', deg=1)
    fit = xr.polyval(group.time, p.polyfit_coefficients)
   # fit_umslimcat=xr.polyval(group.time, fit_para[month-1])
  #  group_detrend = group - fit + group.mean(dim='time') + fit_umslimcat # detrend WACCM forcing file and add mean
    # this is now for all pressure levels!!! only for above 300 hPa desired
    return group - fit + group.mean(dim='time')  
    
 
waccm_o3_forcing=nc_fid_O3[1]['O3']
waccm_o3_forcing=waccm_o3_forcing.groupby('time.month').apply(detrend)


years= np.linspace(2005,2099,95)

waccm_new=np.array(waccm_o3_forcing.sel(time=waccm_o3_forcing.time.dt.year.isin(years))) + np.array(o3_umslimcat.sel(time=o3_umslimcat.time.dt.year.isin(years)))

waccm_o3_forcing=waccm_o3_forcing.sel(time=waccm_o3_forcing.time.dt.year.isin(years))

waccm_new=xr.DataArray(waccm_new, coords=[waccm_o3_forcing.time, waccm_o3_forcing.lev, waccm_o3_forcing.lat], dims=['time','lev', 'lat'], name='O3')


#below 250 hPa and above 1 hPa, waccm forcing should be the normal (non-adjusted) forcing

waccm_new=waccm_new.where((waccm_new.lev<250) & (waccm_new.lev>1), 0) # set values in troposphere to zero


waccm_below_300=nc_fid_O3[1]['O3'].sel(time=nc_fid_O3[1]['O3'].time.dt.year.isin(years)) # non-adjusted file


waccm_below_300=waccm_below_300.where((waccm_below_300.lev>=250) | (waccm_below_300.lev<1),  0) # set values in non-adjusted file above tropopause and below mesopause to zero



waccm_new_total_umslimcat=waccm_new+waccm_below_300 #total forcing with non-adjusted forcing below 300 hPa


waccm_new_total_umslimcat.to_netcdf("/net/hydro/chemie/mfriedel/Data/WACCM/adjusted_o3_forcing_umslimcat_improved.nc")


In [ ]:
# adapt WACCM to SOCOL CCMI ozone trends

model_levels_waccm=nc_fid_O3[1]['lev'] #waccm model levels

#linearly detrend waccm ozone forcing file for each time of the year

o3_socol=nc_fid_O3[3]['zmo3']
o3_socol=o3_socol.squeeze()
o3_socol=o3_socol.interp(plev=model_levels_waccm*100) #interpolate UMSLIMCAT pressure levels to WACCM model levels


o3_socol_npole=xr.DataArray(o3_socol.sel(lat=90, method='nearest'), dims=['time', 'lev'], coords=[o3_socol.time, o3_socol.lev])
o3_socol_spole=xr.DataArray(o3_socol.sel(lat=-90, method='nearest'), dims=['time', 'lev'], coords=[o3_socol.time, o3_socol.lev])

o3_socol_npole=o3_socol_npole.expand_dims(dim={"lat": [90]}) #add variables at poles
o3_socol_spole=o3_socol_spole.expand_dims(dim={"lat": [-90]})

o3_socol=xr.merge([o3_socol, o3_socol_npole, o3_socol_spole])

o3_socol=o3_socol.to_array()

o3_socol=o3_socol.interp(lat=nc_fid_O3[1].lat)


#o3_socol.mean(dim='time').sel(lev=slice(1,50)).plot()

#o3_plot=o3_socol.sel(time=o3_socol.time.dt.year.isin([2005]))
#o3_plot.sel(lev=5, method='nearest').plot()


#o3_socol=o3_socol.interp(lat=nc_fid_O3[1].lat) #interpolate UMSLIMCAT to the same horizontal grid

# substract mean values
o3_socol=o3_socol.groupby('time.month')-o3_socol.groupby('time.month').mean()

#o3_umslimcat=o3_umslimcat.where(o3_umslimcat.plev>300, other=None, drop=False) # mask values in troposphere
    
def calc_trend(group): 
    p = group.polyfit(dim='time', deg=1)
    fit=xr.polyval(group.time, p.polyfit_coefficients)
    return fit
    
o3_socol=o3_socol.groupby('time.month').apply(calc_trend)


def detrend(group):
    p = group.polyfit(dim='time', deg=1)
    fit = xr.polyval(group.time, p.polyfit_coefficients)
   # fit_umslimcat=xr.polyval(group.time, fit_para[month-1])
  #  group_detrend = group - fit + group.mean(dim='time') + fit_umslimcat # detrend WACCM forcing file and add mean
    # this is now for all pressure levels!!! only for above 300 hPa desired
    return group - fit + group.mean(dim='time')  
    
waccm_o3_forcing=nc_fid_O3[1]['O3']
waccm_o3_forcing=waccm_o3_forcing.groupby('time.month').apply(detrend)

years= np.linspace(2005,2099,95)

waccm_new=np.array(waccm_o3_forcing.sel(time=waccm_o3_forcing.time.dt.year.isin(years))) + np.array(o3_socol.sel(time=o3_socol.time.dt.year.isin(years)).squeeze())


waccm_o3_forcing=waccm_o3_forcing.sel(time=waccm_o3_forcing.time.dt.year.isin(years))

waccm_new=waccm_new.squeeze()

waccm_new=xr.DataArray(waccm_new, coords=[waccm_o3_forcing.time, waccm_o3_forcing.lev, waccm_o3_forcing.lat], dims=['time','lev', 'lat'], name='O3')

#below 250 hPa, waccm forcing should be the normal (non-adjusted) forcing

waccm_new=waccm_new.where((waccm_new.lev<250) & (waccm_new.lev>1), 0) # set values in troposphere to zero
#waccm_new=waccm_new.where((waccm_new.lat<87) & (waccm_new.lat>-87), 0) # set values at poles to zero (as SOCOL does not have data there)

waccm_below_300=nc_fid_O3[1]['O3'].sel(time=nc_fid_O3[1]['O3'].time.dt.year.isin(years)) # non-adjusted file

waccm_alt=waccm_below_300.where((waccm_below_300.lev>=250) | (waccm_below_300.lev<=1),  0) # set values in non-adjusted file above tropopause and below mesopause to zero

#waccm_poles=waccm_below_300.where((waccm_below_300.lat<-87) | (waccm_below_300.lat>87),  0) # set values in non-adjusted file above tropopause and below mesopause to zero

waccm_new_total_socol=waccm_new+waccm_alt#total forcing with non-adjusted forcing below 300 hPa and at the poles

waccm_new_total_socol.to_netcdf("/net/hydro/chemie/mfriedel/Data/WACCM/adjusted_o3_forcing_socolccmi_improved.nc")

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(10,5), constrained_layout=True, edgecolor='k',sharex='col')

waccm_100_umslimcat=waccm_new_total_umslimcat.sel(lev=40, method='nearest')
waccm_100_socol=waccm_new_total_socol.sel(lev=40, method='nearest')

waccm_old=O3_col[1].sel(lev=40, method='nearest')
waccm_old.plot(color='blue', linestyle=':', label='WACCM old')

waccm_100_umslimcat=sel_spring((calc_polcap(waccm_100_umslimcat)))
waccm_100_socol=sel_spring((calc_polcap(waccm_100_socol)))

print(waccm_100_umslimcat.sel(year=np.linspace(2005,2019,15)).mean()-waccm_100_umslimcat.sel(year=np.linspace(2085,2099,15)).mean())
print(waccm_100_socol.sel(year=np.linspace(2005,2019,15)).mean()-waccm_100_socol.sel(year=np.linspace(2085,2099,15)).mean())
print(waccm_old.sel(year=np.linspace(2005,2019,15)).mean()-waccm_old.sel(year=np.linspace(2085,2099,15)).mean())
          

plot=waccm_100_umslimcat.plot(color='blue', label='WACCM new UMSLIMCAT')
plot=waccm_100_socol.plot(color='blue', label='WACCM new SOCOL', linestyle='--')
  
o3_umslimcat_100=o3_umslimcat_plot.sel(lev=40, method='nearest')
o3_umslimcat_100.plot(color='green', label='UMSLIMCAT')
o3_socol_100=o3_socol_plot.sel(lev=40, method='nearest')
o3_socol_100.plot(color='orange', label='SOCOL CCMI')

slope, intercept, r_trends, p, se = stats.linregress(o3_umslimcat_100.year, o3_umslimcat_100)
x=np.linspace(o3_umslimcat_100.year.min(), o3_umslimcat_100.year.max(), 100)
plt.plot(x, x*slope +intercept, color='green', linestyle='-')


slope, intercept, r_trends, p, se = stats.linregress(o3_socol_100.year, o3_socol_100)
x=np.linspace(o3_socol_100.year.min(), o3_socol_100.year.max(), 100)
plt.plot(x, x*slope +intercept, color='orange', linestyle='-')


slope, intercept, r_trends, p, se = stats.linregress(waccm_100_umslimcat.year, waccm_100_umslimcat)
x=np.linspace(waccm_100_umslimcat.year.min(), waccm_100_umslimcat.year.max(), 100)
plt.plot(x, x*slope +intercept, color='blue', linestyle='-')
           
slope, intercept, r_trends, p, se = stats.linregress(waccm_100_socol.year, waccm_100_socol)
x=np.linspace(waccm_100_socol.year.min(), waccm_100_socol.year.max(), 100)
plt.plot(x, x*slope +intercept, color='blue', linestyle='-')

slope, intercept, r_trends, p, se = stats.linregress(waccm_old.year, waccm_old)
x=np.linspace(waccm_old.year.min(), waccm_old.year.max(), 100)
plt.plot(x, x*slope +intercept, color='blue', linestyle=':')        
        
plt.legend()

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(10,5), constrained_layout=True, edgecolor='k',sharex='col')

waccm_100_umslimcat=waccm_new_total_umslimcat.sel(lev=50, method='nearest')
waccm_100_socol=waccm_new_total_socol.sel(lev=50, method='nearest')

waccm_old=O3_col[1].sel(lev=50, method='nearest')
waccm_old.plot(color='blue', linestyle=':', label='WACCM old')

waccm_100_umslimcat=sel_spring((calc_polcap(waccm_100_umslimcat)))
waccm_100_socol=sel_spring((calc_polcap(waccm_100_socol)))

print(waccm_100_umslimcat.sel(year=np.linspace(2005,2019,15)).mean()-waccm_100_umslimcat.sel(year=np.linspace(2085,2099,15)).mean())

print(waccm_old.sel(year=np.linspace(2005,2019,15)).mean()-waccm_old.sel(year=np.linspace(2085,2099,15)).mean())
          

plot=waccm_100_umslimcat.plot(color='blue', label='WACCM new UMSLIMCAT')

o3_umslimcat_100=o3_umslimcat_plot.sel(lev=50, method='nearest')
o3_umslimcat_100.plot(color='green', label='UMSLIMCAT')


slope, intercept, r_trends, p, se = stats.linregress(o3_umslimcat_100.year, o3_umslimcat_100)
x=np.linspace(o3_umslimcat_100.year.min(), o3_umslimcat_100.year.max(), 100)
plt.plot(x, x*slope +intercept, color='green', linestyle='-')




slope, intercept, r_trends, p, se = stats.linregress(waccm_100_umslimcat.year, waccm_100_umslimcat)
x=np.linspace(waccm_100_umslimcat.year.min(), waccm_100_umslimcat.year.max(), 100)
plt.plot(x, x*slope +intercept, color='blue', linestyle='-')

slope, intercept, r_trends, p, se = stats.linregress(waccm_old.year, waccm_old)
x=np.linspace(waccm_old.year.min(), waccm_old.year.max(), 100)
plt.plot(x, x*slope +intercept, color='blue', linestyle=':')        
        
plt.legend()

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(10,5), constrained_layout=True, edgecolor='k',sharex='col')

waccm_100_umslimcat=waccm_new_total_umslimcat.sel(lev=50, method='nearest')
waccm_100_socol=waccm_new_total_socol.sel(lev=50, method='nearest')

waccm_old=O3_col[1].sel(lev=50, method='nearest')
waccm_old.plot(color='blue', linestyle=':', label='WACCM old')

waccm_100_umslimcat=sel_spring((calc_polcap(waccm_100_umslimcat)))
waccm_100_socol=sel_spring((calc_polcap(waccm_100_socol)))


print(waccm_100_socol.sel(year=np.linspace(2005,2019,15)).mean()-waccm_100_socol.sel(year=np.linspace(2085,2099,15)).mean())
print(waccm_old.sel(year=np.linspace(2005,2019,15)).mean()-waccm_old.sel(year=np.linspace(2085,2099,15)).mean())
          


plot=waccm_100_socol.plot(color='blue', label='WACCM new SOCOL')

o3_socol_100=o3_socol_plot.sel(lev=50, method='nearest')
o3_socol_100.plot(color='orange', label='SOCOL CCMI')



slope, intercept, r_trends, p, se = stats.linregress(o3_socol_100.year, o3_socol_100)
x=np.linspace(o3_socol_100.year.min(), o3_socol_100.year.max(), 100)
plt.plot(x, x*slope +intercept, color='orange', linestyle='-')

slope, intercept, r_trends, p, se = stats.linregress(waccm_100_socol.year, waccm_100_socol)
x=np.linspace(waccm_100_socol.year.min(), waccm_100_socol.year.max(), 100)
plt.plot(x, x*slope +intercept, color='blue', linestyle='-')

slope, intercept, r_trends, p, se = stats.linregress(waccm_old.year, waccm_old)
x=np.linspace(waccm_old.year.min(), waccm_old.year.max(), 100)
plt.plot(x, x*slope +intercept, color='blue', linestyle=':')        
        
plt.legend()

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(10,5), constrained_layout=True, edgecolor='k',sharex='col')

waccm_100_umslimcat=waccm_new_total_umslimcat.sel(lev=0.1, method='nearest')
waccm_100_socol=waccm_new_total_socol.sel(lev=0.1, method='nearest')

waccm_old=O3_col[1].sel(lev=0.1, method='nearest')
waccm_old.plot(color='blue', linestyle=':', label='WACCM old')

waccm_100_umslimcat=sel_spring((calc_polcap(waccm_100_umslimcat)))
waccm_100_socol=sel_spring((calc_polcap(waccm_100_socol)))

print(waccm_100_umslimcat.sel(year=np.linspace(2005,2019,15)).mean()-waccm_100_umslimcat.sel(year=np.linspace(2085,2099,15)).mean())
print(waccm_100_socol.sel(year=np.linspace(2005,2019,15)).mean()-waccm_100_socol.sel(year=np.linspace(2085,2099,15)).mean())
print(waccm_old.sel(year=np.linspace(2005,2019,15)).mean()-waccm_old.sel(year=np.linspace(2085,2099,15)).mean())
          


slope, intercept, r_trends, p, se = stats.linregress(waccm_100_umslimcat.year, waccm_100_umslimcat)
x=np.linspace(waccm_100_umslimcat.year.min(), waccm_100_umslimcat.year.max(), 100)
plt.plot(x, x*slope +intercept, color='green', linestyle='-', label='adjusted UMSLIMCAT')
           
slope, intercept, r_trends, p, se = stats.linregress(waccm_100_socol.year, waccm_100_socol)
x=np.linspace(waccm_100_socol.year.min(), waccm_100_socol.year.max(), 100)
plt.plot(x, x*slope +intercept, color='orange', linestyle='-', label='adjusted SOCOL')

slope, intercept, r_trends, p, se = stats.linregress(waccm_old.year, waccm_old)
x=np.linspace(waccm_old.year.min(), waccm_old.year.max(), 100)
plt.plot(x, x*slope +intercept, color='blue', linestyle=':')        
        
plt.legend()